[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManzarIMalik/in2stem-placement/blob/main/Intro_to_Computer_Science_and_AI.ipynb)

# What is Computer Science

Computer science is the study of problems, and of what it takes to solve them with a machine.

**It is not really about computers:** the computer is the telescope, not the astronomy. Studying computer science by studying computers is like studying astronomy by studying telescopes - you would learn a lot about lenses and miss the stars entirely.
**It is about problems:** what can be computed, how fast, using how much memory, and what happens when a problem is too big to brute-force.
**It is about precision:** a computer does exactly what you say, which means you have to know exactly what you mean. Most of the difficulty in programming is discovering that you did not.
**It is everywhere:** route-finding in maps, the compression that lets a song be 4 MB instead of 40, the check that your bank card is real, the model that suggested this video.

This notebook is the foundation the other notebooks in this repo stand on. We'll start from the bottom - the fact that *everything* inside a computer is a number - and build up through algorithms, data structures, and the branches the field splits into. That path leads us to one branch in particular: artificial intelligence. We'll trace AI from a 1950 question about whether machines can think, through two collapses of the field, to the generative models people use today, building the key ideas from scratch as we go. There are interactive widgets and "try it yourself" exercises throughout, because you'll remember what you poke at far better than what you read.

In [ ]:
# Colab already ships with NumPy, Matplotlib, scikit-learn, and ipywidgets - nothing to install for this section.
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("NumPy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)
print("Ready to go.")

# Core Concepts in Computer Science

## Everything Is Numbers

Here is the single most important thing to understand about computers: **there is nothing inside but numbers**. No text, no pictures, no sound. A computer stores a number, and something *elsewhere* decides how to interpret it.

Even the numbers are not really numbers - they're **bits**, switches that are either off (0) or on (1). Everything else is a convention built on top of that. Let's look at the actual bits.

In [ ]:
# A number in the base we use, and the base the machine uses.
n = 65
print("Decimal:", n)
print("Binary: ", bin(n))          # 0b prefix just means "this is binary"
print("Binary, padded to 8 bits:", format(n, "08b"))

# Counting to 8 in binary. Notice each column is worth double the one to its right.
print()
for i in range(9):
    print(f"{i:2d}  ->  {i:04b}")

So a byte (8 bits) can hold 256 different patterns, `00000000` through `11111111`. That's the whole vocabulary.

Now the convention: **ASCII** says the number 65 means the letter "A". Not because 65 *is* an "A" in any deep sense - because a committee in 1963 said so and everyone agreed. `ord()` goes from character to number, `chr()` goes back.

In [ ]:
# Text is a lookup table away from being numbers.
for ch in "Hello":
    print(f"{ch}  ->  {ord(ch):3d}  ->  {ord(ch):08b}")

print()
print("And back again:", "".join(chr(n) for n in [83, 97, 108, 102, 111, 114, 100]))

In [ ]:
def text_to_bits(text):
    """Converts a string into a single string of 0s and 1s, 8 bits per character."""
    return " ".join(format(ord(ch), "08b") for ch in text)

def bits_to_text(bits):
    """Converts a space-separated string of 8-bit chunks back into text."""
    return "".join(chr(int(chunk, 2)) for chunk in bits.split())

message = "University of Salford"
encoded = text_to_bits(message)

print("Message:", message)
print("As bits:", encoded)
print("Decoded:", bits_to_text(encoded))
print()
print(f"That message is {len(message) * 8} bits, or {len(message)} bytes.")

**Try it yourself:** encode your own name, then change a *single* bit in the encoded string (a `0` to a `1`) and decode it again. Which letter changed, and by how much? Does flipping the leftmost bit of a chunk do more damage than flipping the rightmost one, and why?

In [ ]:
my_message = "your name here"  # TODO: put your own name in

bits = text_to_bits(my_message)
print("Original:", bits_to_text(bits))

def flip_bit(bits, position):
    """Flips one 0 or 1 in the encoded string, refusing to touch the separators."""
    if bits[position] == " ":
        raise ValueError(
            f"Position {position} is a space between chunks, not a bit. Each letter is "
            f"8 bits wide, so the separators sit at 8, 17, 26, ... Pick another position."
        )
    return bits[:position] + ("1" if bits[position] == "0" else "0") + bits[position + 1:]

# TODO: flip one bit below. Position 0 is the leftmost bit of the first letter,
# position 7 the rightmost. Try both, and compare the damage.
position = 3
print("Corrupted:", bits_to_text(flip_bit(bits, position)))

Here's where it gets interesting. If text is just an array of numbers, and images are just arrays of numbers, and sound is just an array of numbers, then **the array does not know what it is**. The same data can be looked at in completely different ways.

Below we build one NumPy array of numbers and then interpret it twice: once as a picture, once as a sound wave.

In [ ]:
# One array of numbers, two interpretations.
x = np.linspace(0, 4 * np.pi, 400)
wave = np.sin(x) * np.sin(x / 7)          # just numbers between -1 and 1

# Interpretation 1: a sound. Height of the wave over time = air pressure at your eardrum.
# Interpretation 2: an image. Stack the wave into rows and read the numbers as brightness.
image = np.outer(np.sin(x[:200] / 3), wave[:200])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x, wave, linewidth=1)
axes[0].set_title("Read as sound: a waveform")
axes[0].set_xlabel("time")
axes[0].set_ylabel("amplitude")

axes[1].imshow(image, cmap="viridis")
axes[1].set_title("Read as an image: brightness")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("The array itself:", wave[:5], "...")
print("It is not a sound or a picture. It is numbers. We decided what it was.")

This is the idea `Intro_to_computer_vision.ipynb` opens with when it says images are just numbers, and the idea `Intro_to_NLP.ipynb` depends on when it turns words into vectors. Once *everything* is numbers, a single kind of machine - one that does arithmetic very fast - can work on text, photos, music, and video with the same hardware. That universality is why computers ate the world, and it's why one AI architecture can be pointed at pictures *and* language.

## Algorithms: Recipes for Computers

An **algorithm** is a precise sequence of steps that solves a problem. That's all. A recipe is an algorithm, so is long division, so are the instructions for tying a shoelace.

What computer science adds is a question the recipe book never asks: **how does the cost grow as the problem gets bigger?** A recipe for 4 people versus 400 people is roughly 100 times the work. But some algorithms are much cleverer than that, and some are catastrophically worse. This is the difference between a program that finishes and one that does not.

Let's take the simplest possible problem - find a number in a list - and solve it two ways.

In [ ]:
def linear_search(items, target):
    """Checks every item from the start until it finds the target. Returns steps taken."""
    steps = 0
    for i, item in enumerate(items):
        steps += 1
        if item == target:
            return i, steps
    return -1, steps

def binary_search(items, target):
    """Halves a sorted list repeatedly to find the target. Returns steps taken."""
    low, high, steps = 0, len(items) - 1, 0
    while low <= high:
        steps += 1
        mid = (low + high) // 2
        if items[mid] == target:
            return mid, steps
        if items[mid] < target:
            low = mid + 1      # target is in the top half, throw away the bottom
        else:
            high = mid - 1     # target is in the bottom half, throw away the top
    return -1, steps

data = list(range(1_000_000))   # a sorted list of a million numbers
target = 999_999                # the worst case for linear search: the very last one

_, linear_steps = linear_search(data, target)
_, binary_steps = binary_search(data, target)

print(f"Linear search took {linear_steps:,} steps.")
print(f"Binary search took {binary_steps:,} steps.")
print(f"Binary search was {linear_steps // binary_steps:,}x fewer steps.")

Twenty steps instead of a million. Binary search wins because every single comparison throws away *half* of everything still left. It only works on a **sorted** list, which is the catch - that's the price of admission.

Steps are the honest measure, but let's confirm with a stopwatch that this is real time, not a technicality.

In [ ]:
import time

def race(n):
    """Times linear vs binary search for the worst case on a sorted list of n items."""
    items = list(range(n))
    target = n - 1

    start = time.perf_counter()
    linear_search(items, target)
    linear_time = time.perf_counter() - start

    start = time.perf_counter()
    binary_search(items, target)
    binary_time = time.perf_counter() - start

    return linear_time, binary_time

for n in [1_000, 10_000, 100_000, 1_000_000]:
    lt, bt = race(n)
    print(f"n = {n:9,}  |  linear {lt * 1000:9.3f} ms  |  binary {bt * 1000:7.4f} ms")

print()
print("Look down the linear column: 10x the data is roughly 10x the time.")
print("Look down the binary column: 10x the data barely moves it.")

That pattern has a name. We describe the *shape* of the growth and ignore the details, because the shape is what decides whether your program survives contact with real data. This is **Big-O notation**:

* **O(1), constant** - the size of the problem does not matter at all. Looking up a word in a dictionary by its exact key.
* **O(log n), logarithmic** - doubling the data adds one step. Binary search. Extraordinarily good.
* **O(n), linear** - doubling the data doubles the work. Linear search, reading a file.
* **O(n log n)** - the realistic best case for sorting. Python's `sorted()` lives here.
* **O(n squared), quadratic** - doubling the data *quadruples* the work. Comparing every item against every other item. Fine for 100 items, fatal for a million.
* **O(2 to the n), exponential** - adding *one* item doubles the work. Effectively unsolvable beyond tiny inputs, and the reason some problems get attacked with approximations and AI instead of exact answers.

The plot below is worth more than the list. Watch how violently they separate.

In [ ]:
n = np.linspace(1, 100, 500)

plt.figure(figsize=(9, 5.5))
plt.plot(n, np.ones_like(n), label="O(1) constant")
plt.plot(n, np.log2(n), label="O(log n) binary search")
plt.plot(n, n, label="O(n) linear search")
plt.plot(n, n * np.log2(n), label="O(n log n) good sorting")
plt.plot(n, n ** 2, label="O(n squared) compare everything to everything")

plt.ylim(0, 1000)          # without this cap, n squared makes everything else look flat
plt.xlabel("n (size of the problem)")
plt.ylabel("steps taken")
plt.title("How badly does it get worse as the problem grows")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Try it yourself:** before you run the cell below, *predict* the answer out loud. A quadratic algorithm takes 1 second on 1,000 items. How long on 10,000? On 100,000? Then run it and see whether your instinct for growth is calibrated. Would you be happy waiting for a million?

In [ ]:
# A quadratic algorithm taking 1 second on 1,000 items. How long for bigger inputs?
baseline_n = 1_000
baseline_seconds = 1.0

for n in [1_000, 10_000, 100_000, 1_000_000]:  # TODO: add 10_000_000 and brace yourself
    seconds = baseline_seconds * (n / baseline_n) ** 2
    if seconds < 60:
        pretty = f"{seconds:,.0f} seconds"
    elif seconds < 3600:
        pretty = f"{seconds / 60:,.1f} minutes"
    elif seconds < 86400:
        pretty = f"{seconds / 3600:,.1f} hours"
    else:
        pretty = f"{seconds / 86400:,.1f} days"
    print(f"n = {n:12,}  ->  {pretty}")

## Data Structures: Where You Put Things Matters

An algorithm is *how* you do something. A **data structure** is *how you arrange the data* before you start. The two are joined at the hip: the right arrangement makes an algorithm trivial, and the wrong one makes it impossible.

The main characters in Python:

* **List** - an ordered row of things. Great for keeping order and going through everything. Bad at answering "is this in here?" because it has to look at each item.
* **Dictionary** - pairs a key with a value. Answers "what is stored under this key?" in **O(1)**, essentially instantly, no matter how much is in it. This is close to magic and it powers databases, caches, and the word counts in `Intro_to_NLP.ipynb`.
* **Set** - like a dictionary with keys but no values. Instant "have I seen this before?", and it silently refuses duplicates.
* **Stack** - last in, first out, like a stack of plates. Your browser's back button and the "call stack" in a Python error message.
* **Queue** - first in, first out, like a queue for a bus. Print jobs, task schedulers, and the maze solver we'll build later.

In [ ]:
# The same question - "is 999,999 in here?" - asked of two different arrangements.
n = 1_000_000
as_list = list(range(n))
as_dict = {i: True for i in range(n)}
as_set = set(range(n))

target = n - 1

start = time.perf_counter()
target in as_list
list_time = time.perf_counter() - start

start = time.perf_counter()
target in as_dict
dict_time = time.perf_counter() - start

start = time.perf_counter()
target in as_set
set_time = time.perf_counter() - start

print(f"'in' a list: {list_time * 1_000_000:10.1f} microseconds")
print(f"'in' a dict: {dict_time * 1_000_000:10.1f} microseconds")
print(f"'in' a set:  {set_time * 1_000_000:10.1f} microseconds")
print()
print(f"The dict was about {list_time / dict_time:,.0f}x faster.")
print("Same data. Same question. Same computer. Different arrangement.")

The dictionary is not searching. It does arithmetic on the key to work out where the value *must* be, then goes straight there. That trick is called **hashing**, and it's one of the great ideas in computing: instead of looking for something, arrange things so you never have to look.

Stacks and queues are simpler but decide the *order* work gets done, which turns out to matter enormously.

In [ ]:
from collections import deque

# A stack: last in, first out. Think browser history.
stack = []
for page in ["google.com", "wikipedia.org", "salford.ac.uk"]:
    stack.append(page)
    print("Visited:", page)

print()
print("Pressing back:", stack.pop())   # the most recent page comes off first
print("Pressing back:", stack.pop())
print("Now showing: ", stack[-1])

print()
# A queue: first in, first out. Think a print queue, or a bus stop.
queue = deque(["Aisha", "Ben", "Chen"])
print("Queue:", list(queue))
print("Serving:", queue.popleft())     # the person who waited longest goes first
print("Serving:", queue.popleft())
print("Still waiting:", list(queue))

**Try it yourself:** the code below counts how many *distinct* words a text contains, using a list to remember what it has seen. It works, but it's O(n squared) - for every word it scans the whole list of words seen so far. Rewrite it with a `set` or a `dict` and time both on a big text. How much faster is it, and does the gap grow as the text gets longer?

In [ ]:
import random

def make_text(n_words, seed=0):
    """Builds a text whose vocabulary grows with its length, the way a real
    document's does. That growth is what makes the list version below hurt."""
    rng = random.Random(seed)
    vocabulary = [f"word{i}" for i in range(n_words // 10)]
    return [rng.choice(vocabulary) for _ in range(n_words)]

text = make_text(20_000)
print(f"{len(text):,} words, {len(set(text)):,} of them distinct.")

def count_with_list(words):
    """Counts distinct words using a list to track what has been seen. Slow on purpose."""
    seen = []
    for word in words:
        if word not in seen:      # this scans the whole list every single time
            seen.append(word)
    return len(seen)

start = time.perf_counter()
result = count_with_list(text)
slow_time = time.perf_counter() - start
print(f"List version:  {result:,} distinct words in {slow_time * 1000:,.0f} ms")

# TODO: write count_with_set(words) using a set instead, time it the same way,
# and print how many times faster it was. Then double the 20_000 above and
# re-run both. Which one suffers more, and by how much?

## Abstraction and Decomposition

No human being understands all of a modern computer. Not one. The person writing an app does not know how the transistors switch, and the person designing the transistors does not know what app you're building. Software is the largest thing humans construct, and we manage it with two ideas.

**Decomposition** is breaking a problem you cannot solve into problems you can. **Abstraction** is then hiding each solution behind a name, so the next person up can *use* it without *understanding* it.

You did this already. When you called `sorted()`, you did not implement Timsort. You trusted a name. Every layer of a computer is someone doing that to the layer below.

In [ ]:
# Every one of these lines is a real layer, and each one only talks to its neighbour.
layers = [
    "You clicking a button",
    "The app's code (Python, JavaScript)",
    "Libraries the app imports (NumPy, PyTorch)",
    "The language runtime / interpreter",
    "The operating system (Linux, Windows, macOS)",
    "Machine code instructions (ADD, JUMP, LOAD)",
    "The CPU's circuits, logic gates",
    "Transistors - switches made of silicon",
    "Electrons",
]

print("THE STACK OF ABSTRACTIONS")
print("=" * 46)
for depth, layer in enumerate(layers):
    print(f"{'  ' * depth}{'^' if depth else ' '} {layer}")
print("=" * 46)
print("\nEach layer trusts the one below and hides it from the one above.")
print("That trust is the only reason anything this big can be built.")

In [ ]:
# Decomposition in miniature: "is this a valid password" is vague and hard.
# Split it into questions that are each obviously easy, then hide them behind one name.

def is_long_enough(pw):
    """Returns True if the password is at least 8 characters."""
    return len(pw) >= 8

def has_a_number(pw):
    """Returns True if the password contains at least one digit."""
    return any(ch.isdigit() for ch in pw)

def has_mixed_case(pw):
    """Returns True if the password has both upper and lower case letters."""
    return any(ch.isupper() for ch in pw) and any(ch.islower() for ch in pw)

def is_strong(pw):
    """Returns True if a password passes every individual check."""
    return is_long_enough(pw) and has_a_number(pw) and has_mixed_case(pw)

for pw in ["cat", "password", "password1", "Password1", "In2STEM2026"]:
    print(f"{pw:14} -> {'strong' if is_strong(pw) else 'weak'}")

print()
print("is_strong() reads like English. The difficulty went into the small functions,")
print("and now nobody using is_strong() ever has to think about it again.")

## How a Computer Actually Runs Your Code

Your Python is not what runs. The CPU has never heard of Python. It knows a few hundred extremely dull instructions: load a number, add two numbers, compare them, jump somewhere else. Python gets translated down into something like that list.

We can look. `dis` is Python's disassembler, and it shows the **bytecode** your function really became.

In [ ]:
import dis

def add_and_double(a, b):
    total = a + b
    return total * 2

print("The Python you wrote:")
print("    total = a + b")
print("    return total * 2")
print()
print("What actually runs:")
dis.dis(add_and_double)

Two lines of Python became a handful of dumb little steps: push `a`, push `b`, add them, store the result, push it back, push 2, multiply, return. No creativity, no understanding. Just that, a few billion times a second.

The hardware those steps run on is three things you should keep straight, because their speed differences are enormous and they explain a lot of real-world engineering:

* **CPU** - does the arithmetic. Fast beyond intuition: billions of instructions a second.
* **RAM (memory)** - the workspace. Fast, roughly a hundred times slower than the CPU's own registers, and **wiped when the power goes off**.
* **Disk (storage)** - permanent, and dramatically slower again. This is why loading a game takes time but playing it does not.

A **GPU** is the fourth character, and the reason the AI half of this notebook exists at all. A CPU has a few cores that are each brilliant at doing different things quickly. A GPU has thousands of simple cores that all do the *same* arithmetic at once. That's useless for most programs and perfect for neural networks, which are mostly one operation - multiply a big grid of numbers by another big grid - repeated endlessly. Hold that thought until we reach 2012 on the timeline.

> **Going further:** if the layers below Python interest you, the best possible next step is Nand2Tetris, which walks you from a single logic gate up to a working computer running a game you wrote. See [nand2tetris.org](https://www.nand2tetris.org/). For the algorithms and Big-O side, Khan Academy's [algorithms course](https://www.khanacademy.org/computing/computer-science/algorithms) is free and genuinely good.

# Branches of Computer Science

"Computer science" is about as specific as "science". Nobody does all of it. The field has split into branches that share the foundations you just met and then head off in completely different directions - some are almost pure mathematics, some are closer to psychology, some are practically electrical engineering.

Here's the map. Every branch below is a real thing people are paid to do, and every one of them is a route out of this notebook.

In [ ]:
BRANCHES = {
    "Theory and Algorithms": {
        "studies": "What can be computed at all, and what it costs. The mathematics underneath everything else.",
        "question": "Is there a fast algorithm for this, or can we prove there isn't?",
        "job": "Research Scientist, Quantitative Analyst",
        "example": "Proving that some problems cannot be solved quickly is what keeps your bank details encrypted.",
    },
    "Systems and Architecture": {
        "studies": "How the machine itself works: CPUs, memory, operating systems, compilers.",
        "question": "How do we make the hardware fast and the software able to use it?",
        "job": "Systems Engineer, Compiler Engineer",
        "example": "The reason your laptop can run 200 processes on 8 cores without noticing.",
    },
    "Networks": {
        "studies": "Getting data from one machine to another, reliably, over an unreliable world.",
        "question": "How does a message survive a journey through equipment nobody controls?",
        "job": "Network Engineer, Site Reliability Engineer",
        "example": "This notebook travelled through roughly 15 machines to reach you.",
    },
    "Databases": {
        "studies": "Storing enormous amounts of data and answering questions about it fast.",
        "question": "How do we find one row among a billion, and never lose it?",
        "job": "Database Administrator, Data Engineer",
        "example": "Every bank balance, every booking, every login you've ever done.",
    },
    "Security and Cryptography": {
        "studies": "Keeping systems working when someone actively wants them to fail.",
        "question": "How do two strangers share a secret over a wire everyone can listen to?",
        "job": "Security Analyst, Penetration Tester",
        "example": "The padlock in your browser bar is 1970s mathematics doing its job.",
    },
    "Graphics and Visualisation": {
        "studies": "Turning numbers into images: rendering, geometry, simulation, light.",
        "question": "How do we fake reality convincingly, 60 times a second?",
        "job": "Graphics Programmer, Technical Artist",
        "example": "Every game, every Pixar frame, every architectural walkthrough.",
    },
    "Human-Computer Interaction": {
        "studies": "The bit where the computer meets an actual person, who has opinions.",
        "question": "Why did the user do that, and whose fault is it? (Usually the designer's.)",
        "job": "UX Researcher, Interaction Designer",
        "example": "A hospital system that causes fewer errors saves lives without a single new algorithm.",
    },
    "Software Engineering": {
        "studies": "How to build big software with other people without it collapsing.",
        "question": "How does a codebase survive 10 years and 200 contributors?",
        "job": "Software Engineer, DevOps Engineer",
        "example": "The Linux kernel has 30 million lines and thousands of authors. It still boots.",
    },
    "Data Science": {
        "studies": "Getting honest answers out of messy real-world data.",
        "question": "What is this data actually telling us, and where is it lying?",
        "job": "Data Scientist, Data Analyst",
        "example": "See Intro_to_Data_Science_with_Python.ipynb in this repo.",
    },
    "Artificial Intelligence": {
        "studies": "Getting machines to do things we cannot write the rules for.",
        "question": "Can it learn the rules from examples instead of being told them?",
        "job": "Machine Learning Engineer, Research Scientist",
        "example": "This is the branch we walk into for the rest of this notebook.",
    },
}

print(f"{len(BRANCHES)} branches loaded. The dropdown below is built from this dictionary.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def draw_field_map():
    """Draws the branches of computer science as a radial map around a shared core."""
    names = list(BRANCHES.keys())
    # Start at the top and go clockwise, so the map reads like a clock face.
    angles = np.pi / 2 - np.linspace(0, 2 * np.pi, len(names), endpoint=False)

    fig, ax = plt.subplots(figsize=(11, 8.5))
    for angle, name in zip(angles, names):
        x, y = np.cos(angle), np.sin(angle)
        is_ai = name == "Artificial Intelligence"
        colour = "#d94801" if is_ai else "#4292c6"

        ax.plot([0, x], [0, y], color="0.8", linewidth=1, zorder=1)
        ax.scatter([x], [y], s=340 if is_ai else 240, color=colour,
                   edgecolor="black" if is_ai else "white",
                   linewidth=1.6, zorder=3)

        # Labels sit outside the ring rather than inside the dots, so long
        # names like "Human-Computer Interaction" have room to be read.
        ha = "center" if abs(x) < 0.25 else ("left" if x > 0 else "right")
        va = "center" if abs(y) < 0.9 else ("bottom" if y > 0 else "top")
        ax.text(x * 1.16, y * 1.16, name.replace(" and ", "\nand "),
                ha=ha, va=va, fontsize=9.5,
                color="#8c2d04" if is_ai else "#252525",
                fontweight="bold" if is_ai else "normal",
                linespacing=1.35, zorder=3)

    ax.scatter([0], [0], s=5600, color="#252525", zorder=2)
    ax.text(0, 0, "Computer\nScience", ha="center", va="center",
            fontsize=12, color="white", fontweight="bold", zorder=3)

    ax.set_xlim(-1.95, 1.95)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("The branches of computer science (highlighted: where we go next)",
                 fontsize=13, pad=10)
    plt.tight_layout()
    plt.show()

draw_field_map()

In [ ]:
from ipywidgets import interact, Dropdown

def explore_branch(branch):
    """Prints what a branch of computer science studies, and what you'd do in it."""
    info = BRANCHES[branch]
    print(branch.upper())
    print("=" * len(branch))
    print(f"\nStudies:        {info['studies']}")
    print(f"\nCore question:  {info['question']}")
    print(f"\nJob titles:     {info['job']}")
    print(f"\nIn the wild:    {info['example']}")

interact(explore_branch, branch=Dropdown(options=list(BRANCHES.keys()),
                                         value="Artificial Intelligence",
                                         description="Branch:"));

Two things worth noticing about that map.

The branches are not walls. Modern AI is completely dependent on **systems** (someone has to make 10,000 GPUs behave), **databases** (the training data has to live somewhere), **security** (models get attacked), and **HCI** (a model nobody can use is worth nothing). The interesting work is usually where two branches touch.

And AI is *not* the whole of computer science, despite what the news implies. It's one branch, it's had a spectacular decade, and it's had spectacular decades before that were followed by collapses. We're about to look at exactly that.

**Try it yourself:** add a branch to the `BRANCHES` dictionary that isn't there - quantum computing, bioinformatics, robotics, computer music, whatever you'd actually want to do - then re-run the map and the dropdown. Both are built from the dictionary, so both update. Why is that a better design than typing the branch into the drawing code?

In [ ]:
# TODO: add your own branch. Copy the shape of the entries above.
BRANCHES["Your Branch Here"] = {
    "studies": "",
    "question": "",
    "job": "",
    "example": "",
}

# TODO: uncomment these once you've filled it in
# draw_field_map()
# interact(explore_branch, branch=Dropdown(options=list(BRANCHES.keys()), description="Branch:"));

print("Branches now on the map:", len(BRANCHES))

# Artificial Intelligence

## What Makes a Problem an AI Problem

Forget the films. Here is the working definition, and it's less glamorous and far more useful:

**AI is for problems where we cannot write down the rules.**

That's it. That's the line. Everything you wrote in `Learning_Python_In2Stem_Placement.ipynb` was you knowing the rules and telling the computer. Some problems are not like that - not because they're hard, but because *you* can do them effortlessly and still cannot explain how.

In [ ]:
# Problem 1: is this number even? You know the rule. You can write it in one line.
def is_even(n):
    """Returns True if a number is even. The rule is known, exact, and complete."""
    return n % 2 == 0

print("is_even(4):", is_even(4))
print("is_even(7):", is_even(7))
print("Perfect. Correct for every number that will ever exist. Done forever.")
print()

# Problem 2: is this photo a cat? Try to write the rule.
def is_cat(photo):
    """Returns True if the photo contains a cat. Go on, then."""
    # if it has pointy ears... but so do foxes, and some cats have folded ears
    # if it has whiskers... you cannot see whiskers on a dark cat at low resolution
    # if it has fur... so does the carpet it's sitting on
    # if it has four legs... and so does the chair behind it
    raise NotImplementedError("Nobody has ever written this function. Not once.")

print("You have recognised cats since you were two years old.")
print("You have never once known the rule.")

Sit with that for a second, because it's the whole reason this field exists. You can recognise a friend's face in a crowd, in bad light, at an angle, after five years. You cannot write down how. The knowledge is real and it is *not available to you in the form of rules*.

This is **Polanyi's paradox**: we know more than we can tell. And it's a wall for traditional programming, because traditional programming requires you to tell.

So AI takes the other road. Instead of writing rules, **show the machine thousands of examples and let it work the rules out itself.** That's the whole idea. Everything after this - the winters, the neural networks, ChatGPT - is that one bet being made with different amounts of data, compute, and cleverness.

Notice what this buys and what it costs. You get to solve problems you cannot explain. You give up knowing exactly what the thing learned, whether it's right, and why it did that. Every controversy about AI traces back to that trade.

## A Timeline of AI, 1950 to Today

AI is not new. It is older than the microprocessor, older than the internet, older than nearly everyone reading this. It has been "about to change everything" roughly four times, and twice it collapsed so badly that researchers hid the words "artificial intelligence" from their grant applications.

That history is the best inoculation against the current hype in either direction. Let's draw it.

In [ ]:
ERAS = {
    "1950 - The Question": {
        "years": (1950, 1955), "level": 0.35,
        "what": "Alan Turing publishes 'Computing Machinery and Intelligence', opening with 'Can machines think?'",
        "why": "He replaces an unanswerable philosophical question with a testable one: can a machine hold a conversation well enough that you cannot tell? The Imitation Game, now the Turing test. The field has a target before it has a computer worth running it on.",
    },
    "1956 - The Birth": {
        "years": (1956, 1960), "level": 0.55,
        "what": "The Dartmouth Workshop. McCarthy, Minsky, Shannon and others spend a summer on it and coin the term 'artificial intelligence'.",
        "why": "Their proposal claimed a significant advance could be made in one summer by ten people. That optimism is a running theme. The field gets its name here.",
    },
    "1957 - The Perceptron": {
        "years": (1957, 1965), "level": 0.75,
        "what": "Frank Rosenblatt builds the perceptron, a machine that learns from examples rather than being programmed.",
        "why": "The first real neural network, built in hardware. The New York Times reported it would walk, talk, see, write and reproduce itself. It was one neuron. We build one from scratch later in this notebook.",
    },
    "1966 - ELIZA": {
        "years": (1966, 1972), "level": 0.5,
        "what": "Joseph Weizenbaum writes ELIZA, a 200-line program that reflects your words back as questions like a therapist.",
        "why": "People confided in it. Weizenbaum's own secretary asked him to leave the room. It understood nothing, and it fooled people anyway - the first hard evidence that we read minds into machines that do not have them. Worth remembering when a chatbot seems to like you.",
    },
    "1969 - The First Wall": {
        "years": (1969, 1973), "level": 0.3,
        "what": "Minsky and Papert's book 'Perceptrons' proves a single perceptron cannot learn XOR, an almost trivial function.",
        "why": "Devastating and technically correct. Funding for neural networks evaporated for over a decade. The fix - stacking layers - was already imaginable, but nobody knew how to train the stack yet. You will make this exact machine fail, on purpose, later.",
    },
    "1974 - First AI Winter": {
        "years": (1974, 1980), "level": 0.12,
        "what": "The 1973 Lighthill report savages the field's progress. UK and US funding is cut hard.",
        "why": "The promises of the 60s met reality: the demos did not scale, and the computers were pitiful. 'AI' became a word that cost you your grant. The first winter.",
    },
    "1980s - Expert Systems": {
        "years": (1980, 1987), "level": 0.6,
        "what": "Rule-based expert systems reach industry. XCON saves DEC an estimated $40m a year by configuring computer orders.",
        "why": "AI's first real commercial success, and it was pure hand-written rules - no learning at all. You'll build a small one in the next section, and then break it.",
    },
    "1987 - Second AI Winter": {
        "years": (1987, 1993), "level": 0.1,
        "what": "The expert system market collapses. Specialised AI hardware is wiped out by ordinary desktop computers.",
        "why": "Rules do not scale. Every exception needs a new rule, the rules start contradicting each other, and only the original author can maintain them. The lesson: hand-written knowledge hits a ceiling, hard.",
    },
    "1997 - Deep Blue": {
        "years": (1997, 2005), "level": 0.55,
        "what": "IBM's Deep Blue beats world chess champion Garry Kasparov.",
        "why": "A milestone and a disappointment at once. It won by searching 200 million positions a second - brute force plus expert tuning, not understanding. It taught us that beating humans at a task does not require doing it the way humans do.",
    },
    "2012 - Deep Learning": {
        "years": (2012, 2016), "level": 0.8,
        "what": "AlexNet wins the ImageNet competition by a colossal margin using a deep neural network on two gaming GPUs.",
        "why": "The big bang. The ideas were 1980s ideas - what was new was ImageNet's millions of labelled images and GPUs to crunch them. Neural networks go from disreputable to dominant in about a year. This is where Intro_to_computer_vision.ipynb starts working.",
    },
    "2016 - AlphaGo": {
        "years": (2016, 2017), "level": 0.85,
        "what": "DeepMind's AlphaGo beats Lee Sedol at Go, a game with more legal positions than atoms in the observable universe.",
        "why": "Go was thought to be decades away because brute force is hopeless there. AlphaGo learned intuition from self-play. Move 37 of game two was so alien that commentators assumed a bug, and it was brilliant.",
    },
    "2017 - Transformers": {
        "years": (2017, 2019), "level": 0.9,
        "what": "Eight researchers at Google publish 'Attention Is All You Need', introducing the Transformer.",
        "why": "The most consequential AI paper of the century so far. It let models read a whole sentence at once instead of word by word, which meant training could be parallelised across thousands of GPUs. Every model in the rest of this notebook is a Transformer.",
    },
    "2018 - Scaling Up": {
        "years": (2018, 2021), "level": 0.92,
        "what": "BERT, then GPT-2, then GPT-3. Models grow from millions to 175 billion parameters.",
        "why": "The shocking discovery: just making it bigger kept working, and abilities nobody trained for started appearing. GPT-3 could translate and do arithmetic having only ever been asked to predict the next word.",
    },
    "2022 - Generative AI": {
        "years": (2022, 2023), "level": 0.95,
        "what": "Stable Diffusion and DALL-E 2 generate images from text. ChatGPT reaches 100 million users in two months.",
        "why": "The year AI stopped being an industry story. The technology was not new - the interface was. Anyone could type a sentence and get a result, and the whole world found out at once.",
    },
    "2023-2026 - Multimodal and Agents": {
        "years": (2023, 2026), "level": 0.97,
        "what": "Models handle text, images, audio and video together, use tools, write and run code, and are trained to reason step by step before answering.",
        "why": "The frontier as you read this. The open questions are not really technical any more: reliability, cost, energy, bias, jobs, and whether the scaling that got us here keeps paying. You are arriving at a genuinely undecided moment.",
    },
}

print(f"{len(ERAS)} eras. Now let's draw them.")

In [ ]:
def draw_timeline():
    """Draws the history of AI as a timeline, with the two AI winters shaded."""
    fig, ax = plt.subplots(figsize=(15, 7.5))

    # The winters first, so everything else sits on top of them.
    for start, end, label in [(1974, 1980, "First AI Winter"), (1987, 1993, "Second AI Winter")]:
        ax.axvspan(start, end, color="#c6dbef", alpha=0.7, zorder=0)
        ax.text((start + end) / 2, -0.055, label, ha="center", fontsize=9,
                style="italic", color="#08306b", zorder=1)

    ax.axhline(0, color="0.3", linewidth=1.5, zorder=1)

    # The recent eras are only a year or two apart, so labels stacked straight
    # above the dots would sit on top of each other. Give those six their own
    # slots out to the side and run a thin leader line back to the dot.
    LABEL_SLOTS = {
        "2012 - Deep Learning": (2006.0, 1.02, "right"),
        "2016 - AlphaGo": (2011.5, 1.17, "center"),
        "2017 - Transformers": (2015.5, 1.32, "center"),
        "2018 - Scaling Up": (2021.5, 1.32, "center"),
        "2022 - Generative AI": (2026.5, 1.17, "center"),
        "2023-2026 - Multimodal and Agents": (2027.5, 1.02, "left"),
    }

    for name, era in ERAS.items():
        year = era["years"][0]
        level = era["level"]
        winter = "Winter" in name
        colour = "#6baed6" if winter else ("#d94801" if level > 0.75 else "#2171b5")

        ax.plot([year, year], [0, level], color=colour, linewidth=1.6, zorder=2)
        ax.scatter([year], [level], s=90, color=colour, zorder=3)

        head, tail = name.split(" - ", 1)
        if name in LABEL_SLOTS:
            lx, ly, ha = LABEL_SLOTS[name]
            # Aim the leader at the text's near edge, never across its middle.
            edge = {"left": -0.3, "right": 0.3, "center": 0.0}[ha]
            ax.annotate("", xy=(year, level + 0.02), xytext=(lx + edge, ly - 0.02),
                        arrowprops=dict(arrowstyle="-", color="0.6",
                                        linewidth=0.9, shrinkA=6, shrinkB=2),
                        zorder=2)
        else:
            lx, ly, ha = year, level + 0.055, "center"

        ax.text(lx, ly, head, ha=ha, va="bottom", fontsize=9,
                fontweight="bold", zorder=4)
        ax.text(lx, ly - 0.012, tail, ha=ha, va="top", fontsize=7.5,
                color="0.35", zorder=4)

    ax.set_xlim(1946, 2035)
    ax.set_ylim(-0.09, 1.5)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_yticks([])
    ax.set_title("A history of artificial intelligence, 1950 to 2026\n"
                 "(height is roughly how excited the world was, which is not the same as progress)",
                 fontsize=12.5, pad=12)
    for spine in ["left", "right", "top"]:
        ax.spines[spine].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

draw_timeline()

Read the shape of that plot, not just the labels. It goes up, it *crashes*, it goes up higher, it crashes again, and then from 2012 it climbs to somewhere it has never been. The two blue valleys are the winters - periods when the money, the jobs, and the credibility went away.

Every peak before 2012 was followed by a fall, and every fall came from the same thing: the demo worked, and then it did not scale. That's the pattern to keep in your head while you read the news.

Use the dropdown to walk through each era properly.

In [ ]:
from ipywidgets import interact, Dropdown

def explore_era(era):
    """Prints what happened in a given era of AI history and why it mattered."""
    info = ERAS[era]
    start, end = info["years"]
    print(era.upper())
    print("=" * len(era))
    print(f"\nPeriod:  {start} to {end}")
    print(f"\nWhat happened:\n  {info['what']}")
    print(f"\nWhy it mattered:\n  {info['why']}")

interact(explore_era, era=Dropdown(options=list(ERAS.keys()), description="Era:",
                                   layout={"width": "460px"}));

**Try it yourself:** work out how old each breakthrough is. Print each era with its age in years as of today, and find the gap between the perceptron (1957) and deep learning finally working (2012). The core idea sat there, mostly unusable, for how long? What does that suggest about which ideas are "dead" right now?

In [ ]:
THIS_YEAR = 2026  # TODO: update if you're reading this later

for name, era in ERAS.items():
    year = era["years"][0]
    print(f"{name:34}  {year}   {THIS_YEAR - year:3d} years ago")

# TODO: print the gap between Rosenblatt's perceptron and AlexNet.
# The idea did not change much in that time. What changed?

## Symbolic AI: Teaching by Rules

For AI's first thirty years, nearly everyone believed intelligence *was* symbol manipulation. Get the knowledge out of an expert's head, write it as rules, and the machine reasons with it. This is **symbolic AI**, or **good old-fashioned AI**, and it produced the field's first commercial hit - the expert systems on the timeline at 1980.

The idea is completely reasonable. Let's build one. This is a triage system, of the kind that really was deployed in the 80s.

In [ ]:
def triage(symptoms):
    """A tiny expert system: maps a set of symptoms to advice using hand-written rules."""
    s = set(symptoms)

    if "chest pain" in s and "shortness of breath" in s:
        return "EMERGENCY: call 999 immediately."
    if "high fever" in s and "stiff neck" in s and "rash" in s:
        return "EMERGENCY: possible meningitis, go to A&E."
    if "high fever" in s and "cough" in s:
        return "See a GP today."
    if "cough" in s and "runny nose" in s:
        return "Likely a cold. Rest and fluids."
    if "headache" in s:
        return "Rest, water, and paracetamol if needed."
    return "No rule matched. Please see a GP."

cases = [
    ["chest pain", "shortness of breath"],
    ["high fever", "cough"],
    ["cough", "runny nose"],
    ["headache"],
]

for case in cases:
    print(f"{str(case):48} -> {triage(case)}")

That works. It's fast, it's free, it runs on a calculator, and - this matters enormously - **you can read exactly why it said what it said**. Point at the line. That property is called interpretability, and modern neural networks have almost none of it. In medicine and law, that's not a small thing.

So why did the entire expert systems industry collapse in 1987? Watch.

In [ ]:
# Reality arrives with cases nobody wrote a rule for.
awkward_cases = [
    (["chest pain"], "Just chest pain. Serious or indigestion?"),
    (["high fever", "cough", "chest pain"], "Overlaps two rules. Which one wins?"),
    (["tired", "thirsty", "losing weight"], "Textbook diabetes. No rule exists."),
    (["cough", "runny nose", "chest pain", "shortness of breath"], "A cold AND an emergency."),
]

for symptoms, note in awkward_cases:
    print(f"{str(symptoms):58}")
    print(f"  system says: {triage(symptoms)}")
    print(f"  problem:     {note}")
    print()

Look at the last one carefully. The system says "EMERGENCY", which is right, but only by luck - the emergency rule happened to be written first. Rule order is doing medicine. Nobody decided that.

This is the ceiling that killed symbolic AI, and it's worth naming precisely:

* **The rules never end.** Every real case reveals another exception. XCON at DEC reached about 10,000 rules and needed a full-time team just to keep it standing.
* **Rules collide.** Once two rules match, something has to break the tie, and that tiebreak is usually accidental.
* **The knowledge has to exist in words first.** Back to Polanyi: the expert cannot tell you how they know. They just know.
* **It cannot generalise.** Nothing in `triage()` has any idea what a fever *is*. Shown a symptom one word off, it's helpless.

**Try it yourself:** add a diabetes rule for tired plus thirsty plus losing weight. Now add a rule for a chest infection (cough plus high fever plus chest pain). Does it interfere with the emergency rule? Keep adding rules for the awkward cases above. How many rules before you're not sure what your own system will do?

In [ ]:
def triage_v2(symptoms):
    """Your improved expert system. Add rules until you lose confidence in it."""
    s = set(symptoms)

    # TODO: add a diabetes rule: tired + thirsty + losing weight
    # TODO: add a chest infection rule: cough + high fever + chest pain
    # TODO: does your new rule fire before or after the emergency rule? Does that matter?

    if "chest pain" in s and "shortness of breath" in s:
        return "EMERGENCY: call 999 immediately."
    if "high fever" in s and "cough" in s:
        return "See a GP today."
    return "No rule matched. Please see a GP."

print(triage_v2(["tired", "thirsty", "losing weight"]))

Hold on to the feeling of that exercise, because it is *precisely* the feeling the whole field had by 1987. The rules never end. And the escape route is the one we've been building towards: stop writing rules, show it examples instead.

## Search as Intelligence

Before we get to learning, there's a second classical idea that produced AI's most famous victory: **search**. Don't know the answer? Try possibilities systematically until you find it.

It sounds like cheating. It won at chess. Deep Blue beat Kasparov in 1997 by examining 200 million positions a second - it had no idea what a "plan" was, it just looked further ahead than a human can.

Let's watch search actually think. Here's a maze, solved by **breadth-first search**, which explores everywhere one step away, then everywhere two steps away, and so on - using the queue from earlier.

In [ ]:
from collections import deque

MAZE = [
    "S...#.....",
    ".##.#.###.",
    ".#..#...#.",
    ".#.###.##.",
    ".#.....#..",
    ".####.##.#",
    "....#..#..",
    "###.#.##.#",
    "....#....#",
    ".##....#.E",
]

def solve_maze(maze):
    """Finds the shortest path with breadth-first search. Returns the path and every cell visited."""
    rows, cols = len(maze), len(maze[0])
    start = end = None
    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == "S":
                start = (r, c)
            elif maze[r][c] == "E":
                end = (r, c)

    queue = deque([start])          # cells we know about but haven't explored
    came_from = {start: None}       # how we reached each cell
    visited_order = []

    while queue:
        current = queue.popleft()   # FIFO: this is what makes it breadth-first
        visited_order.append(current)
        if current == end:
            break
        r, c = current
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (r + dr, c + dc)
            nr, nc = nxt
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] != "#" and nxt not in came_from:
                came_from[nxt] = current
                queue.append(nxt)

    if end not in came_from:
        return [], visited_order

    path, node = [], end            # walk backwards from the end to rebuild the route
    while node is not None:
        path.append(node)
        node = came_from[node]
    return path[::-1], visited_order

path, visited = solve_maze(MAZE)
print(f"Cells in the maze:     {sum(row.count('.') for row in MAZE) + 2}")
print(f"Cells the search visited: {len(visited)}")
print(f"Length of shortest path:  {len(path)}")

In [ ]:
def draw_maze(maze, path, visited):
    """Draws the maze, every cell the search explored, and the shortest path it found."""
    rows, cols = len(maze), len(maze[0])
    grid = np.zeros((rows, cols))
    for r in range(rows):
        for c in range(cols):
            grid[r, c] = 0 if maze[r][c] == "#" else 1

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    for ax, title in zip(axes, ["Everywhere the search looked", "The shortest path it found"]):
        ax.imshow(grid, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=12)
        ax.set_xticks([]); ax.set_yticks([])

    # Left: the search order, coloured by when each cell was reached.
    vr = [p[0] for p in visited]
    vc = [p[1] for p in visited]
    axes[0].scatter(vc, vr, c=range(len(visited)), cmap="autumn", s=110, marker="s")

    # Right: just the answer.
    pr = [p[0] for p in path]
    pc = [p[1] for p in path]
    axes[1].plot(pc, pr, color="#d94801", linewidth=3.5)
    axes[1].scatter([pc[0]], [pr[0]], color="green", s=180, zorder=3, label="Start")
    axes[1].scatter([pc[-1]], [pr[-1]], color="red", s=180, zorder=3, label="End")
    axes[1].legend(loc="upper right")

    plt.tight_layout()
    plt.show()

draw_maze(MAZE, path, visited)
print("Left: the thinking. Right: the answer.")
print("It found the shortest route without any idea what a maze is.")

The left panel is the point. The search had no insight, no intuition, no plan - it flooded outwards checking everything until it fell over the exit. It still found the *provably shortest* path. That's real, useful problem-solving from pure mechanism, and it's genuinely how your maps app routes you home (with smarter variants called A* and Dijkstra's algorithm).

So why isn't this AI solved? **Combinatorial explosion** - the O(2 to the n) monster from the Big-O list. Look at what happens when the board gets bigger than a maze.

In [ ]:
# Why search alone cannot get you to intelligence.
games = [
    ("Noughts and crosses", 5_478),
    ("Chess (Deep Blue, 1997)", 10 ** 47),
    ("Go (AlphaGo, 2016)", 10 ** 170),
]

atoms_in_universe = 10 ** 80

for name, positions in games:
    pretty = f"{positions:,}" if positions < 10 ** 6 else f"{positions:.0e}"
    print(f"{name}")
    print(f"  {pretty} legal board positions")
    if positions > atoms_in_universe:
        print(f"  that is more than the {atoms_in_universe:.0e} atoms in the observable universe")
    print()

print("Deep Blue searched 200,000,000 positions per second and won at chess.")
print("Do the arithmetic for Go and you get a number with no physical meaning.")
print("You cannot brute-force your way there. AlphaGo had to learn judgement instead.")

And there it is - the wall that ends the classical era. Rules do not scale (1987). Search does not scale (Go). Both approaches shared one assumption: that intelligence is something you *build*.

The alternative is that intelligence is something you *grow*, from data.

## The AI Winters

Two things on that timeline were labelled winters, and they're the most useful part of this history, so let's be explicit about the anatomy. Both times, the cycle ran like this:

1. **A genuine breakthrough.** The perceptron really did learn. Expert systems really did save DEC $40m a year. These were not frauds.
2. **Extrapolation from the demo.** Rosenblatt's one neuron gets reported as a machine that will "walk, talk, see and reproduce itself". A demo on a toy problem is treated as a solved problem awaiting engineering.
3. **The demo does not scale.** The toy problem was not a small version of the real one. The perceptron cannot do XOR. Rule 10,001 contradicts rule 47.
4. **The money leaves all at once.** Funders don't gently taper, they exit. The Lighthill report in 1973 emptied UK AI funding almost overnight. Researchers rebranded as "informatics" and "machine learning" to survive.

Now the uncomfortable question, which nobody can answer for you: **is the current boom different?**

The honest case for yes: unlike every previous peak, this one is generating enormous revenue from products millions of people use daily. Scaling has kept working far past where everyone expected it to break. The results are not toy demos.

The honest case for no: models still confidently invent facts, cost staggering sums to train, need energy at the scale of small countries, and the datasets are close to exhausted - we're running out of internet. Several of the specific promises being made right now (full self-driving "next year", every year, since 2016) look a lot like step 2.

Nobody in 1985 knew they were 18 months from a winter. Hold both cases at once, and be suspicious of anyone who is certain in either direction.

> **Going further:** the primary sources are more readable than you'd expect. Turing's 1950 paper [Computing Machinery and Intelligence](https://academic.oup.com/mind/article/LIX/236/433/986238) is genuinely funny in places and he answers objections people still raise today. The [Dartmouth proposal](http://www-formal.stanford.edu/jmc/history/dartmouth/dartmouth.html) is two pages and its confidence is something to behold.

# Machine Learning

## Learning From Data Instead of Rules

**Machine learning is the part of AI where the machine finds the rules itself, from examples.**

You do not write `is_cat()`. You collect 10,000 photos labelled cat or not-cat, and you run a procedure that adjusts numbers until the answers come out right. Nobody ever specifies what a cat looks like. That knowledge ends up inside the model as a pile of numbers no human can read, which is both the trick and the catch.

Machine learning comes in three flavours, and knowing which one you're looking at tells you most of what you need:

* **Supervised learning** - you have examples *with the right answers attached*. Photos labelled cat/dog, emails labelled spam/not-spam, houses with their sale prices. The model learns the mapping from question to answer. This is the overwhelming majority of ML in production, and it's what we'll do below.
* **Unsupervised learning** - examples with *no* answers. You're asking the model to find structure you didn't know was there: which customers behave alike, which of these transactions is weird. Nobody says what the groups should be.
* **Reinforcement learning** - no answers, just a score. The model acts, gets rewarded or punished, and works out a strategy over thousands of attempts. This is how AlphaGo learned Go by playing itself millions of times, and it's how models get tuned to be helpful in conversation.

Let's do supervised learning properly, right now, on real data.

## Your First Model

We'll use handwritten digits - the actual problem that sorted post through mail centres for decades. 1,797 images, each 8x8 pixels, each labelled with the digit it shows.

First rule of machine learning, before any code: **look at your data**.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()

print("Images:  ", digits.images.shape, " (1797 images, each 8 rows x 8 columns)")
print("Labels:  ", digits.target.shape)
print("Classes: ", np.unique(digits.target))

fig, axes = plt.subplots(2, 8, figsize=(13, 3.6))
for ax, image, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"{label}", fontsize=11)
    ax.axis("off")
plt.suptitle("The data: handwritten digits, as a human sees them", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# And now as the model sees them. Remember: everything is numbers.
image = digits.images[0]

print("What you see: a handwritten zero")
print("What the model gets:\n")
for row in image:
    print("  " + " ".join(f"{int(v):2d}" for v in row))

print("\nEach number is how dark that pixel is, 0 (white) to 16 (black).")
print("Flattened into one long row, that is 64 numbers:")
print(digits.data[0].astype(int))
print("\nThat row of 64 numbers is the entire input. No eyes, no intuition.")

Now the single most important habit in the entire field: **split the data before you train**.

We hide a chunk of the digits from the model completely. It trains on the rest. Then we test it on the chunk it has never seen. Why? Because a model that memorises every answer scores 100% on data it has memorised and is *worthless*. The only score that means anything is the score on data it has never laid eyes on.

Getting this wrong is the most common way real machine learning projects quietly fail.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = digits.data, digits.target   # X is what we know, y is what we want to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training on {len(X_train)} digits.")
print(f"Holding back {len(X_test)} digits the model will never see during training.")

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)                     # this line is the learning

print(f"\nAccuracy on data it trained on:  {model.score(X_train, y_train):.1%}")
print(f"Accuracy on data it has NEVER seen: {model.score(X_test, y_test):.1%}")
print("\nThat second number is the only one that counts.")

Around 97% on digits it has never seen, from about ten lines of code, and at no point did anyone describe what a 7 looks like. Compare that with the effort of writing `triage()`, which handled four cases and broke on the fifth.

That is the whole argument for machine learning in one comparison.

But accuracy is a single number, and single numbers hide things. *Which* digits does it get wrong? A **confusion matrix** shows every mistake, and the mistakes are where the understanding is.

In [ ]:
from sklearn.metrics import confusion_matrix

predictions = model.predict(X_test)
cm = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap="Blues")

for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=9)

ax.set_xlabel("What the model predicted", fontsize=11)
ax.set_ylabel("What it actually was", fontsize=11)
ax.set_title("Confusion matrix: the diagonal is correct, everything else is a mistake", fontsize=12)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

errors = [(i, j, cm[i, j]) for i in range(10) for j in range(10) if i != j and cm[i, j] > 0]
errors.sort(key=lambda e: -e[2])
print("Its most common mistakes:")
for actual, predicted, count in errors[:5]:
    print(f"  thought a {actual} was a {predicted}  ({count} times)")

In [ ]:
# Let's actually look at the ones it got wrong. This is more informative than any metric.
wrong = np.where(predictions != y_test)[0]
print(f"It got {len(wrong)} of {len(y_test)} wrong. Here are some:\n")

fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(13, 2.4))
for ax, idx in zip(np.atleast_1d(axes), wrong[:8]):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"said {predictions[idx]}\nwas {y_test[idx]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Be honest: could you have read all of those correctly at 8x8?")

## Features: What the Model Actually Sees

The model got 64 numbers per image. Those numbers are called **features**, and choosing them used to be the entire job.

Before 2012, a computer vision researcher's career was spent hand-designing features: edge detectors, corner detectors, texture descriptors, clever mathematical summaries of an image. The machine learning bit on the end was almost an afterthought. Getting the features right was where the skill and the PhDs went.

The revolution of deep learning - the 2012 spike on our timeline - is that **the model learns the features too**. You hand it raw pixels and it figures out for itself that edges matter, then that corners matter, then that eyes matter. We'll see that happen with our own eyes in a few cells.

Here's why features matter so much: the same data, described differently, can turn an impossible problem into a trivial one.

In [ ]:
# Two rings of points. Can you separate them with a straight line? No. Nobody can.
rng = np.random.RandomState(0)
n = 300
angle = rng.uniform(0, 2 * np.pi, n)
radius = np.concatenate([rng.uniform(0, 1.2, n // 2), rng.uniform(2.2, 3.2, n // 2)])
x1 = radius * np.cos(angle)
x2 = radius * np.sin(angle)
labels = np.concatenate([np.zeros(n // 2), np.ones(n // 2)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(x1, x2, c=labels, cmap="coolwarm", s=22)
axes[0].set_title("Original features (x, y)\nNo straight line can separate these", fontsize=11)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

# One new feature: distance from the centre. That is all it takes.
distance = np.sqrt(x1 ** 2 + x2 ** 2)
axes[1].scatter(distance, rng.uniform(0, 1, n), c=labels, cmap="coolwarm", s=22)
axes[1].axvline(1.7, color="black", linestyle="--", linewidth=2)
axes[1].set_title("New feature: distance from centre\nNow a child could draw the line", fontsize=11)
axes[1].set_xlabel("distance from centre"); axes[1].set_yticks([])

plt.tight_layout()
plt.show()

print("Same data. Same points. Different description.")
print("Impossible problem -> trivial problem, with one good feature.")
print("That is what feature engineering was, and what deep learning now does by itself.")

## Overfitting: The Thing That Ruins Everything

If you remember one idea from this entire notebook, make it this one.

**Overfitting is when a model memorises the training data instead of learning the pattern.** It scores brilliantly on what it has seen and falls apart on anything new. It is the single most common failure in machine learning, it is *invisible* if you don't hold data back, and it has a human equivalent you already know: the student who memorised past papers and is destroyed by a question phrased differently.

Let's cause it deliberately. We'll fit curves of increasing wiggliness to some noisy data.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# The truth is a simple wave. The data is that wave plus noise, because reality is noisy.
rng = np.random.RandomState(3)
X_small = np.sort(rng.uniform(0, 1, 18))[:, None]
y_small = np.sin(2 * np.pi * X_small).ravel() + rng.normal(0, 0.25, 18)
X_grid = np.linspace(0, 1, 300)[:, None]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

for ax, degree, verdict in zip(axes, [1, 4, 17],
                               ["Underfitting: too simple to see the wave",
                                "About right: it found the pattern",
                                "Overfitting: it memorised the noise"]):
    curve = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    curve.fit(X_small, y_small)

    ax.scatter(X_small, y_small, color="black", s=32, zorder=3, label="the data")
    ax.plot(X_grid, np.sin(2 * np.pi * X_grid), "g--", linewidth=1.4, label="the real pattern")
    ax.plot(X_grid, curve.predict(X_grid), color="#d94801", linewidth=2, label="what it learned")
    ax.set_ylim(-2, 2)
    ax.set_title(f"degree {degree}\n{verdict}", fontsize=10.5)
    ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

Look at the right-hand panel. That orange line passes through *almost every single data point*. On the training data it is nearly perfect - better than the middle one by any score you care to compute. And it is garbage. It has learned the noise, the random jitter that will never repeat, and between the points it does things the real pattern never does.

If you only measured training accuracy, you would ship the right-hand model. This is not a hypothetical mistake, it happens constantly.

Now the curve that shows the trap in one picture.

In [ ]:
degrees = range(1, 18)
train_errors, test_errors = [], []

X_tr, X_te, y_tr, y_te = train_test_split(X_small, y_small, test_size=0.4, random_state=1)

for degree in degrees:
    curve = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    curve.fit(X_tr, y_tr)
    train_errors.append(np.mean((curve.predict(X_tr) - y_tr) ** 2))
    test_errors.append(np.mean((curve.predict(X_te) - y_te) ** 2))

plt.figure(figsize=(9.5, 5.5))
plt.plot(degrees, train_errors, "o-", label="Error on data it trained on")
plt.plot(degrees, test_errors, "s-", label="Error on data it has never seen")
sweet_spot = degrees[int(np.argmin(test_errors))]     # let the data say where it is
plt.axvline(sweet_spot, color="green", linestyle="--", alpha=0.7)
plt.text(sweet_spot + 0.15, max(test_errors) * 0.62, "the sweet spot", color="green", fontsize=10)
plt.yscale("log")
plt.xlabel("Model complexity (polynomial degree)")
plt.ylabel("Error (lower is better, log scale)")
plt.title("The most important plot in machine learning")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Training error (blue) falls forever. More complexity always fits the past better.")
print("Test error (orange) falls, bottoms out, then CLIMBS. That climb is overfitting.")
print("Everything to the right of the green line is a model getting worse while looking better.")

The blue line falling forever is the seduction: a more complicated model *always* explains the data you already have more precisely. The orange line turning upward is reality: past a point, it's explaining things that aren't real.

The defences are worth knowing by name, because you'll meet all of them:

* **Hold data back.** Non-negotiable. This is the only reason you can see the orange line at all.
* **More data.** Harder to memorise a million examples than eighteen. This is why the modern models are trained on much of the internet.
* **Simpler models.** If two models do equally well, take the plainer one.
* **Regularisation.** Actively penalise the model for complexity during training.
* **Cross-validation.** Split the data several different ways and average, so you don't get fooled by one lucky split.

**Try it yourself:** change `test_size` in the cell below from 0.4 to 0.2 and then to 0.6, and re-run. Watch where the sweet spot moves. Then increase the amount of data (change `18` to `200` in the data-generating cell above and re-run both). What happens to overfitting when data is plentiful, and what does that tell you about why big models need big data?

In [ ]:
# TODO: change test_size and re-run. Then go back and generate more data.
test_size = 0.4

X_tr, X_te, y_tr, y_te = train_test_split(X_small, y_small, test_size=test_size, random_state=1)

scores = {}
for degree in [1, 3, 6, 12, 17]:
    curve = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    curve.fit(X_tr, y_tr)
    scores[degree] = np.mean((curve.predict(X_te) - y_te) ** 2)

best = min(scores, key=scores.get)
for degree, error in scores.items():
    marker = "  <-- best on unseen data" if degree == best else ""
    print(f"degree {degree:2d}:  test error {error:8.3f}{marker}")

# Neural Networks

## The Perceptron

Time to go back to 1957 and build the thing itself.

Rosenblatt's **perceptron** is one artificial neuron, and it is embarrassingly simple. It:

1. takes some inputs,
2. multiplies each by a **weight** (how much that input matters),
3. adds them up, plus a **bias** (how eager it is to fire at all),
4. outputs 1 if the total clears zero, otherwise 0.

That's the entire machine. The learning is just this: when it gets one wrong, nudge the weights in the direction that would have made it right. Do that a few hundred times.

No libraries for this one. Fifteen lines of NumPy.

In [ ]:
class Perceptron:
    """A single artificial neuron, as Frank Rosenblatt described it in 1957."""

    def __init__(self, n_inputs, learning_rate=0.1):
        self.weights = np.zeros(n_inputs)
        self.bias = 0.0
        self.learning_rate = learning_rate

    def predict(self, x):
        """Weighs the inputs, sums them, and fires 1 if the total clears zero."""
        total = np.dot(x, self.weights) + self.bias
        return 1 if total > 0 else 0

    def train(self, X, y, epochs=20):
        """Nudges the weights towards the right answer every time it gets one wrong."""
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, target in zip(X, y):
                error = target - self.predict(xi)
                if error != 0:
                    self.weights += self.learning_rate * error * xi   # the entire learning rule
                    self.bias += self.learning_rate * error
                    errors += 1
            history.append(errors)
        return history

print("That is a neural network. All of it.")

In [ ]:
# Teach it logical AND: output 1 only when both inputs are 1.
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

neuron = Perceptron(n_inputs=2)
print("Before training (weights are all zero):")
for xi in X_and:
    print(f"  {xi} -> {neuron.predict(xi)}")

history = neuron.train(X_and, y_and, epochs=20)

print(f"\nAfter training (it settled after {next((i for i, e in enumerate(history) if e == 0), '?')} passes):")
for xi, target in zip(X_and, y_and):
    guess = neuron.predict(xi)
    print(f"  {xi} -> {guess}   (wanted {target})  {'correct' if guess == target else 'WRONG'}")

print(f"\nWeights it discovered: {neuron.weights}, bias: {neuron.bias:.2f}")
print("Nobody told it the rule for AND. It worked the numbers out from four examples.")

It learned. Nobody wrote the AND rule - it found weights that implement AND by being corrected four examples at a time.

Now let's do what Minsky and Papert did in 1969, and break it. **XOR** is "exclusive or": true when the inputs are *different*. It is, if anything, simpler to describe than AND.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])            # 1 when the inputs differ

neuron = Perceptron(n_inputs=2)
history = neuron.train(X_xor, y_xor, epochs=500)   # 500 passes. Give it every chance.

print("After 500 training passes:")
for xi, target in zip(X_xor, y_xor):
    guess = neuron.predict(xi)
    print(f"  {xi} -> {guess}   (wanted {target})  {'correct' if guess == target else 'WRONG'}")

print(f"\nCorrections made on each of the last 10 passes: {history[-10:]}")
print("(That counts every nudge it made mid-pass, so it does not match the two")
print(" WRONG above - the weights keep moving as it works through the examples.)")
print("\nIt never gets there. Not in 500 passes, not in 500 million.")
print("This single result froze neural network research for over a decade.")

Now *why*, because the reason is beautiful and it explains what a neuron fundamentally is.

A perceptron computes a weighted sum and asks "is it over the line?" That's not a metaphor. It is literally drawing **one straight line** and calling everything on one side 1. So a perceptron can only solve a problem if a straight line can separate the answers.

Look at what that means for AND versus XOR.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6))

for ax, y_vals, title in zip(axes, [y_and, y_xor], ["AND: one line does it", "XOR: no line exists"]):
    for xi, target in zip(X_and, y_vals):
        ax.scatter(xi[0], xi[1], s=420, marker="o" if target == 1 else "X",
                   color="#d94801" if target == 1 else "#2171b5", zorder=3)
        ax.annotate(f"answer: {target}", (xi[0], xi[1]), textcoords="offset points",
                    xytext=(0, -26), ha="center", fontsize=10, zorder=4)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.6)
    ax.set_xlabel("input 1"); ax.set_ylabel("input 2")
    ax.set_title(title, fontsize=12)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

# AND: a line cleanly separates the single 1 from the three 0s.
xs = np.linspace(-0.5, 1.5, 10)
axes[0].plot(xs, 1.5 - xs, "g--", linewidth=2.2, label="a line that works", zorder=2)
axes[0].legend(loc="upper left", fontsize=9)

# XOR: swing a line through every angle and each one gets a point wrong.
# Parallel lines would only ever test one direction, which proves nothing.
for angle in np.linspace(0, np.pi, 7, endpoint=False):
    dx, dy = np.cos(angle), np.sin(angle)
    # Wobble each pivot off centre, so this reads as "lines everywhere"
    # rather than a starburst through one special point.
    cx = 0.5 + 0.22 * np.cos(2.3 * angle)
    cy = 0.5 + 0.22 * np.sin(1.7 * angle + 0.9)
    t = np.array([-2, 2])
    axes[1].plot(cx + dx * t, cy + dy * t, color="red", linestyle="--",
                 alpha=0.4, linewidth=1.2, zorder=1)
axes[1].plot([], [], color="red", linestyle="--", alpha=0.6, label="every angle fails")
axes[1].legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

print("The 1s in XOR sit diagonally opposite. Get a ruler and try.")
print("A single neuron draws a single line, so a single neuron cannot do XOR. Ever.")

The fix is almost insulting in retrospect: **use more than one neuron**. One neuron draws a line. A layer of neurons draws several lines. A layer on top of that combines those lines into regions, and regions can be any shape at all.

That's what "deep" means in deep learning. Not deep thought - just layers stacked on layers.

So why did the field sit in a winter for a decade if the fix is "add a layer"? Because nobody knew how to *train* the middle layers. If the network gets it wrong, whose fault is it? Which of the hidden neurons should change, and by how much? That's the **credit assignment problem**, and the answer - **backpropagation**, popularised in 1986 - is what ended the first winter.

Let's just check that the fix works.

In [ ]:
from sklearn.neural_network import MLPClassifier

# The same impossible XOR, given to a network with one hidden layer of 4 neurons.
network = MLPClassifier(hidden_layer_sizes=(4,), max_iter=8000, random_state=1)
network.fit(X_xor, y_xor)

print("A single neuron (1957):        cannot do XOR, ever.")
print("Two layers of neurons (1986):")
for xi, target in zip(X_xor, y_xor):
    guess = network.predict([xi])[0]
    print(f"  {xi} -> {guess}   (wanted {target})  {'correct' if guess == target else 'WRONG'}")

print(f"\nAccuracy: {network.score(X_xor, y_xor):.0%}")
print("\nOne extra layer. That is the whole difference between a dead field and deep learning.")

**Try it yourself:** teach the original single `Perceptron` class OR (`[0, 1, 1, 1]`) and NAND (`[1, 1, 1, 0]`). Both should work. Then try to reason about why *before* you run them: sketch the four points and see whether a line separates them. Can you find any other 2-input logic function the perceptron cannot learn?

In [ ]:
# TODO: try OR, then NAND. Predict success or failure before you run it.
y_or = np.array([0, 1, 1, 1])     # TODO: swap in [1, 1, 1, 0] for NAND

neuron = Perceptron(n_inputs=2)
neuron.train(X_and, y_or, epochs=50)

for xi, target in zip(X_and, y_or):
    guess = neuron.predict(xi)
    print(f"  {xi} -> {guess}   (wanted {target})  {'correct' if guess == target else 'WRONG'}")

## Gradient Descent: How Learning Actually Happens

We waved at "nudge the weights in the right direction". Let's look at the nudge, because **gradient descent** is the engine underneath essentially all modern AI - the perceptron, AlexNet, and the model that wrote your last autocomplete were all trained by this one idea.

Picture the model's error as a landscape. Every possible setting of the weights is a location, and the height is how wrong the model is there. Training is finding the bottom of a valley.

The catch, and it's the whole thing: **the model is blind**. It cannot see the landscape - with billions of weights, nobody can. All it can feel is the **slope** under its feet. So it does the only thing available: step downhill, repeatedly, and hope.

The size of that step is the **learning rate**, and it is the most temperamental setting in machine learning. The landscape below has two valleys, one deeper than the other, and the run always starts on the right. Play with it.

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def loss_landscape(w):
    """A pretend error landscape with two valleys: one shallow, one deep."""
    return 0.05 * w ** 4 - 0.5 * w ** 2 + 0.3 * w

def loss_slope(w):
    """The slope of that landscape at w. This is all the model can actually feel."""
    return 0.2 * w ** 3 - w + 0.3

GLOBAL_MIN, LOCAL_MIN = -2.37, 2.07     # where the two valley bottoms sit

def roll_downhill(learning_rate, steps):
    """Runs gradient descent from a fixed start and draws every step it took."""
    w = 4.0                                     # always start on the right-hand slope
    path = [w]
    for _ in range(steps):
        w = w - learning_rate * loss_slope(w)   # THE line. Step against the slope.
        w = np.clip(w, -50, 50)                 # stop the plot exploding if it diverges
        path.append(w)

    grid = np.linspace(-6, 6, 400)
    plt.figure(figsize=(10, 5.5))
    plt.plot(grid, loss_landscape(grid), color="0.6", linewidth=2, label="error landscape")
    plt.plot(path, [loss_landscape(p) for p in path], "o-", color="#d94801",
             markersize=5, linewidth=1.2, alpha=0.85, label="the model's steps")
    plt.scatter([path[0]], [loss_landscape(path[0])], color="green", s=140, zorder=5, label="start")
    plt.scatter([path[-1]], [loss_landscape(path[-1])], color="red", s=140, zorder=5, label="ended up")
    plt.axvline(GLOBAL_MIN, color="green", linestyle=":", alpha=0.6)
    plt.text(GLOBAL_MIN, 6.5, " the deep valley", color="green", fontsize=9)
    plt.axvline(LOCAL_MIN, color="purple", linestyle=":", alpha=0.6)
    plt.text(LOCAL_MIN, 6.5, " the shallow one", color="purple", fontsize=9)
    plt.ylim(-3, 8); plt.xlim(-6, 6)
    plt.xlabel("the weight being learned"); plt.ylabel("how wrong the model is")
    plt.title(f"learning rate = {learning_rate}   |   ended at w = {path[-1]:.2f}, error = {loss_landscape(path[-1]):.2f}")
    plt.legend(loc="upper center"); plt.grid(alpha=0.3)
    plt.show()

    final = path[-1]
    if abs(final) > 20:
        print("DIVERGED. The steps are so big it overshoots further every time.")
        print("The error is now astronomical. This is a training run you would kill.")
    elif abs(final - path[-2]) > 0.01:
        print("Still crawling. The steps are too small - it ran out of time before")
        print("reaching any bottom at all. Give it more steps, or a bigger rate.")
    elif abs(final - LOCAL_MIN) < 0.5:
        print("Settled - but in the SHALLOW valley, and it has no idea.")
        print("It cannot see the deeper valley on the left. The slope under its feet")
        print("is zero, so as far as it knows, the job is done. This is a local minimum.")
    else:
        print("Settled in the deep valley. This is the best answer available,")
        print("and notice it took a BIGGER step to get here - it had to leap over")
        print("the hill in the middle to escape the shallow valley.")

interact(roll_downhill,
         learning_rate=FloatSlider(min=0.01, max=1.5, step=0.01, value=0.2, description="rate:"),
         steps=IntSlider(min=1, max=60, step=1, value=25, description="steps:"));

**Try it yourself:** work through these four settings in order, and watch the message under the plot change.

* **0.02** - too small. It crawls and never arrives, even though it's going the right way. Real training runs die like this, quietly, wasting a week of GPU time.
* **0.2** - it works. It settles at the bottom of a valley in a handful of steps. Look *carefully* at which valley.
* **0.5** - bigger steps, and now something surprising happens. Where does it end up, and why is that *better*?
* **1.0 and up** - catastrophe. Each step overshoots harder than the last.

Now the uncomfortable bit, and it's the reason we built the landscape with two valleys. At 0.2 the model settles in the **shallow** valley and reports itself finished. It is not finished. There's a better answer sitting just over the hill, and the model cannot see it, because all it can feel is the slope right under its feet - and that slope is zero. It has no way to know. That's a **local minimum**, and the blindness is not a bug in our code, it's the actual situation every model is in.

What rescues it here is a *bigger* step, which is deeply counterintuitive: taking cruder steps lets it leap the hill and find the deeper valley. That's genuinely how it works, and it's why learning rates get tuned rather than derived.

So why isn't every neural network hopelessly stuck? Because in the billion-dimensional landscapes of real networks, local minima turn out to matter far less than this two-dimensional picture suggests - with that many directions to move in, it's rare for *every single one* to point uphill. That's one of the lucky facts that made deep learning work at all, and nobody knew it until they tried.

## Why Deep: Seeing What a Network Learns

Let's put it together and train a real (small) network on the digits, then do the thing that makes deep learning click: **look inside it**.

In [ ]:
network = MLPClassifier(hidden_layer_sizes=(64,), max_iter=600, random_state=1)
network.fit(X_train, y_train)

print(f"Single neuron (1957):       cannot even do XOR")
print(f"Logistic regression:        {model.score(X_test, y_test):.1%} on unseen digits")
print(f"Neural network (64 hidden): {network.score(X_test, y_test):.1%} on unseen digits")
print(f"\nWeights in this network: {sum(w.size for w in network.coefs_):,}")
print("GPT-3 has 175,000,000,000. Same idea, more of it.")
print()
print("Be honest about that result: the network barely beat plain logistic")
print("regression. On 8x8 digits there is not much for depth to do. The reason")
print("to care is what it learned on the way, which we can look at.")

In [ ]:
# Each hidden neuron has one weight per input pixel. 64 weights = an 8x8 image.
# So we can literally look at what each neuron is watching for.
fig, axes = plt.subplots(4, 8, figsize=(13, 7))
weights = network.coefs_[0]     # shape: (64 pixels, 64 hidden neurons)

for i, ax in enumerate(axes.ravel()):
    ax.imshow(weights[:, i].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(f"neuron {i}", fontsize=7.5)
    ax.axis("off")

plt.suptitle("What each hidden neuron looks for\n"
             "red = 'ink here excites me', blue = 'ink here puts me off'", fontsize=13)
plt.tight_layout()
plt.show()

print("Nobody designed these. They are what the network invented, by itself,")
print("from nothing but pixels and corrections. Some watch for loops, some for")
print("strokes, some for gaps in particular places.")

Those little patterns are **learned features** - the thing we said used to be a researcher's whole career. The network invented its own edge and stroke detectors because they were useful, without a single instruction about what a digit is.

Now scale that idea up and you have modern computer vision. Stack the layers and each one builds on the last:

* **Layer 1** learns edges and blobs.
* **Layer 2** combines edges into corners and curves.
* **Layer 3** combines those into textures and parts - an eye, a wheel, a letter.
* **Layer 8** combines parts into whole objects - a face, a bus, a cat.

Nobody specified that hierarchy. It emerges, every time, because it's the efficient way to describe the visual world. `Intro_to_computer_vision.ipynb` visualises exactly this in a much bigger network, and you'll recognise it when you see it.

And this is why 2012 happened when it did. The ideas above were all in place by 1986. What arrived in 2012 was **ImageNet** (millions of labelled photos) and **GPUs** (the thousands-of-simple-cores chips from earlier, perfect for multiplying grids of numbers). Same idea, finally with enough data and compute to breathe.

> **Going further:** 3Blue1Brown's [neural networks series](https://www.3blue1brown.com/topics/neural-networks) is the best explanation of backpropagation that exists, and it's free. If you want to feel a network train, [TensorFlow Playground](https://playground.tensorflow.org) lets you build one in a browser and watch the regions form.

# Tasks in AI

Almost everything called "AI" is one of a small number of task shapes wearing different clothes. Learn the shapes and the field gets much less intimidating - a new product announcement usually turns out to be a familiar task on a new kind of data.

* **Classification** - put it in one of N boxes. Spam or not spam, which digit, which of 1,000 objects. What we did with the digits, and what `Intro_to_computer_vision.ipynb` does with YOLO's classifier.
* **Regression** - predict a number. House price, tomorrow's temperature, how long this delivery takes. See `Intro_to_Data_Science_with_Python.ipynb`.
* **Clustering** - find the groups nobody labelled. Which customers behave alike. Unsupervised, so there's no right answer to check against, which makes it much slipperier than it sounds.
* **Detection** - not just *what* but *where*. Boxes round every car in a photo. `Intro_to_computer_vision.ipynb` does this with YOLO11.
* **Segmentation** - which exact *pixels* are the cat. Detection's fussier sibling, and how phone portrait mode blurs your background.
* **Generation** - produce new data that could have been real. Text, images, code, music, video. The section below, and most of what's in the news.
* **Translation and summarisation** - turn one sequence into another. See `Intro_to_NLP.ipynb`.
* **Recommendation** - predict what someone will like. Quietly the most commercially valuable AI on earth, running your feed right now.
* **Reinforcement learning** - learn a strategy from rewards. Game-playing, robotics, and the tuning that makes chatbots agreeable.

Here's the thing worth sitting with: **the boundaries between these are dissolving**. In 2015, each one was a separate research community with its own conferences and architectures. Today a single large model does most of them, because "describe this image" is generation, and "is this spam" is generation if the output you generate is the word "spam". The task shapes are becoming prompts.

# The Modern Era: Transformers and Generative AI

2017, on the timeline. Eight researchers at Google publish a paper with the cheeky title *Attention Is All You Need*, and every AI system you've heard of since is descended from it.

The problem they were solving was mundane. Older language models read a sentence one word at a time, in order, carrying a memory forward - which meant they forgot the start of long sentences, and worse, could not be parallelised. Word 50 needs word 49 first. Ten thousand GPUs cannot help you read a sentence in order.

The **Transformer** let a model look at every word at once and *learn which words to pay attention to*. That killed the bottleneck, training became parallel, and suddenly money and hardware converted directly into capability. Everything since - GPT, BERT, ChatGPT, Claude, image and video models - is that architecture, scaled.

Let's take a real one apart. Small models, CPU only, nothing to sign up for.

In [ ]:
!pip install transformers

## Text Becomes Numbers

Everything is numbers, still. So the first thing any language model must do is turn your sentence into a list of numbers, exactly like ASCII did at the top of this notebook - except the units aren't letters, and they aren't quite words either. They're **tokens**: common chunks of text, learned from data.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

sentence = "Computer science is the study of problems."
token_ids = tokenizer.encode(sentence)
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print("Your sentence:", sentence)
print(f"\nBroken into {len(tokens)} tokens:")
for token, tid in zip(tokens, token_ids):
    print(f"  {token!r:16} -> {tid}")

print("\nWhat the model actually receives:", token_ids)
print("\nThat odd G-with-a-bar character marks a space before the word.")
print("Common words survive as one token, rarer ones get chopped into pieces.")

In [ ]:
# Common words survive whole. Unusual ones get shredded.
for text in ["the cat sat", "Salford", "antidisestablishmentarianism", "strawberry", "1234567"]:
    pieces = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    print(f"{text:32} -> {len(pieces):2d} tokens  {pieces}")

That last one explains a famous embarrassment. Ask a large language model how many r's are in "strawberry" and it has historically got it wrong, which looks like a devastating failure of reasoning.

Look at what it actually receives. It does not see `s-t-r-a-w-b-e-r-r-y`. It sees two or three token ids - arbitrary numbers standing for chunks. **You are asking it to count letters in something it cannot see the letters of.** It's like asking you to count the strokes in a Chinese character you only ever heard spoken aloud.

Not a reasoning failure. A plumbing failure. Most surprising AI limitations turn out to be like this - boring consequences of how the thing is wired, not deep facts about intelligence. Being able to tell those apart is most of what it means to understand this field.

## Attention, Intuitively

Now the idea that named the paper. Take this sentence:

> The trophy would not fit in the suitcase because **it** was too big.

What is "it"? The trophy, obviously. Now change one word:

> The trophy would not fit in the suitcase because **it** was too small.

Now "it" is the suitcase. One adjective, right at the end, flips what a pronoun refers to. No grammar rule catches that. You need to actually *weigh the words against each other*.

**Attention** is exactly that, done with arithmetic: for every word, score how much every other word matters to it, then take a weighted average. That's the whole mechanism. "It" learns to look hard at "trophy", "suitcase", and the adjective on the end.

In [ ]:
# A hand-built attention pattern, to show the shape of the thing.
# (Real ones are learned, and there are dozens per layer, but they look like this.)
words = ["The", "trophy", "would", "not", "fit", "in", "the", "suitcase", "because", "it", "was", "too", "big"]
n = len(words)

rng = np.random.RandomState(7)
attention = rng.uniform(0, 0.12, (n, n))       # a quiet background hum
for i in range(n):
    attention[i, i] = 0.25                     # words attend to themselves

it = words.index("it")
attention[it, words.index("trophy")] = 0.85    # "it" looks hard at the two candidates
attention[it, words.index("suitcase")] = 0.55
attention[it, words.index("big")] = 0.60       # and at the adjective that decides it
attention[words.index("fit"), words.index("trophy")] = 0.5
attention = attention / attention.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(attention, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(words, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(words)
ax.set_xlabel("...pays attention to these words", fontsize=11)
ax.set_ylabel("This word...", fontsize=11)
ax.set_title("Attention: who is looking at whom\n(read the row for 'it')", fontsize=12.5)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Read the 'it' row: it is lit up under 'trophy', 'suitcase' and 'big'.")
print("Swap 'big' for 'small' and a trained model shifts that weight to the suitcase.")

Two consequences of that picture, and they're the reason the world changed in 2017:

**It's parallel.** Every word scores every other word at the same time. No waiting, no reading left to right. That's what let training spread across thousands of GPUs, and it's what turned money into capability.

**It's not language-specific.** Nothing in "score how much each thing matters to each other thing" mentions words. Chop an image into patches and the same machinery works, which is how one architecture ended up doing text, images, audio and video. That's why the field converged so fast.

## Generating Text

Now the part that shocked everyone. A language model does exactly one thing: **predict the next token**. Then it sticks its own prediction on the end and predicts again. That's it. That's the whole trick behind every chatbot you've used.

`distilgpt2` is a small, old, weak model - 82 million parameters against GPT-3's 175 billion, roughly two thousand times smaller. It downloads in seconds and runs on a CPU. It is not going to impress you, and that's useful: you can watch the mechanism without being dazzled.

In [ ]:
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")

# device=-1 means CPU, so this works on a free Colab runtime with no GPU.
generator = pipeline("text-generation", model="distilgpt2", device=-1)

prompt = "Computer science is"
output = generator(prompt, max_new_tokens=30, do_sample=False, pad_token_id=50256)

print("Prompt: ", prompt)
print("Output: ", output[0]["generated_text"])
print("\nIt predicted one token, added it, predicted again, 30 times over.")

In [ ]:
# Let's watch the prediction happen, one token at a time, with the actual probabilities.
import torch
from transformers import AutoModelForCausalLM

model_lm = AutoModelForCausalLM.from_pretrained("distilgpt2")
model_lm.eval()

def show_next_token_options(text, top_k=8):
    """Prints the model's top candidates for the next token, with probabilities."""
    ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        logits = model_lm(ids).logits[0, -1, :]      # scores for every possible next token
    probs = torch.softmax(logits, dim=-1)            # turn scores into probabilities
    top = torch.topk(probs, top_k)

    print(f'"{text}..."')
    print(f"{'next token':>16}   probability")
    for prob, tid in zip(top.values, top.indices):
        token = tokenizer.decode(tid)
        bar = "#" * int(prob.item() * 60)
        print(f"{token!r:>16}   {prob.item():6.1%}  {bar}")
    print()

show_next_token_options("The capital of France is")
show_next_token_options("Machine learning is a")

There is no plan. No sentence it intends to write. Just a probability distribution over what comes next, sampled, appended, repeated.

Which raises the obvious question: if it just picks the likeliest word, why isn't every answer identical and dull? Because we don't have to take the likeliest word. **Temperature** controls how much risk it takes: low temperature always grabs the safest option, high temperature lets long shots win.

This one slider is the difference between a model that repeats itself forever and one that writes nonsense. Go and break it.

In [ ]:
from ipywidgets import interact, FloatSlider, Dropdown

def generate_with_temperature(temperature, prompt):
    """Generates text at a given temperature and comments on what the setting does."""
    if temperature < 0.05:
        output = generator(prompt, max_new_tokens=40, do_sample=False, pad_token_id=50256)
    else:
        output = generator(prompt, max_new_tokens=40, do_sample=True,
                           temperature=temperature, top_k=50, pad_token_id=50256)

    print(f"TEMPERATURE = {temperature}")
    print("=" * 62)
    print(output[0]["generated_text"])
    print()
    if temperature < 0.3:
        print("Safe and repetitive. It always grabs the single likeliest token, so it")
        print("often falls into a loop and says the same thing forever. Notice it gives")
        print("the identical answer every time you run it - at 0.0 there is no randomness.")
    elif temperature < 0.9:
        print("The useful zone: coherent, but it can still surprise you. A model this")
        print("small still loops sometimes - that is distilgpt2 being tiny, not the")
        print("temperature being wrong. A bigger model at 0.7 rarely does this.")
    elif temperature < 1.4:
        print("Getting adventurous. Creative, and starting to wander off the point.")
    else:
        print("Chaos. Long-shot tokens keep winning, so grammar and meaning fall apart.")

interact(generate_with_temperature,
         temperature=FloatSlider(min=0.0, max=2.0, step=0.1, value=0.7, description="temp:"),
         prompt=Dropdown(options=["The best thing about computers is",
                                  "In the year 2050, artificial intelligence will",
                                  "The University of Salford is",
                                  "Once upon a time"],
                         description="prompt:", layout={"width": "470px"}));

**Try it yourself:** run the same prompt at temperature 0.0 five times, then at 1.8 five times. At 0.0 you get the identical output every time (it's not random at all), and at 1.8 you get five different piles of wreckage. Now find the highest temperature that still produces something you'd call a sentence. Where's the edge, and does it move depending on the prompt?

In [ ]:
prompt = "The best thing about computers is"

# TODO: run this cell a few times. Then set temperature to 0.0 and run it again.
# Which setting gives you the same answer every time, and why is that not a bug?
temperature = 1.8

# Temperature 0 is not "a tiny bit of randomness", it is none at all - so we ask
# for greedy decoding rather than a division by zero the library will reject.
sampling = temperature >= 0.05

for i in range(3):
    if sampling:
        out = generator(prompt, max_new_tokens=25, do_sample=True, temperature=temperature,
                        top_k=50, pad_token_id=50256)
    else:
        out = generator(prompt, max_new_tokens=25, do_sample=False, pad_token_id=50256)
    print(f"{i + 1}. {out[0]['generated_text']}\n")

## Predicting the Next Word Is Enough

Here's the genuinely astonishing part, and the bit nobody predicted.

Everyone assumed next-word prediction was a toy objective - useful for autocomplete, irrelevant to intelligence. Then models got big, and abilities **nobody trained for** started showing up. GPT-3 could translate French. Nobody taught it to translate. It could do arithmetic, write code, answer trivia, summarise. It was only ever asked to predict the next token.

Why does that happen? Think about what it takes to *actually* be excellent at the game:

* "The capital of France is ___" - you need facts.
* "The murderer was revealed to be the ___" - you need to have followed the plot.
* "2 + 2 = ___" - you need arithmetic.
* "The French for 'hello' is ___" - you need French.
* "def factorial(n): return ___" - you need to program.

To predict the next word *well enough*, across everything humans have ever written, you have to model a great deal about the world that produced those words. The prediction task is a trapdoor. Squeeze hard enough on it and capability falls out.

This is **emergence**, and it's the central surprise of the last decade: certain abilities are absent, absent, absent at increasing model sizes, and then abruptly present. Not gradually better - a step change, with no warning from the smaller models.

You just watched `distilgpt2`, 82 million parameters, produce mush. The identical mechanism at a thousand times the scale writes working code. Nobody has a satisfying theory of why. That's not a gap in this notebook, it's a gap in the field.

## Beyond Text

The last few years took that machinery and pointed it at everything else. Briefly, because this moves fast enough that anything specific here will age badly:

* **Multimodal models** take text, images, audio and video *together*. Photograph your homework and ask about it. The trick, as we said, is that attention never cared whether the things it was weighing were words.
* **Diffusion models** generate images by starting with pure noise and repeatedly removing a bit of it, steered by your text. This is Stable Diffusion, DALL-E and Midjourney. `Intro_to_computer_vision.ipynb` has a bonus section that runs one on a GPU runtime, if you want to watch it happen.
* **Reasoning models** are trained to write out their thinking before answering, and get much better at maths and logic as a result. The model is not smarter, it's been given room to work.
* **Agents** use tools: search the web, run code, call APIs, act in a loop. The model stops being a thing you talk to and becomes a thing that does. This is where most of the current effort and most of the current failures are.
* **Open models** matter more than the headlines suggest. Llama, Mistral, Qwen, Gemma and thousands of others are free to download and run - `Intro_to_computer_vision.ipynb` uses the [Hugging Face Hub](https://huggingface.co/models) for this. You are not locked out of this technology.

## What These Models Get Wrong

Every honest introduction to this stuff ends here, because the failures are not footnotes. They are properties of how the thing works, and none of them are fixed.

**They make things up.** Not lying - a model has no concept of truth. It generates plausible next tokens, and false things are often extremely plausible. Watch.

In [ ]:
# Ask a model about something that does not exist, and watch it not notice.
fake_prompt = "The Zorbian Theorem, discovered in 1847, states that"

output = generator(fake_prompt, max_new_tokens=40, do_sample=True,
                   temperature=0.7, top_k=50, pad_token_id=50256)

print(output[0]["generated_text"])
print()
print("There is no Zorbian Theorem. It does not exist. It was never discovered.")
print("The model does not know that, cannot know that, and will not hesitate.")
print("It is not lying - it has no concept of truth to lie about. It is doing")
print("exactly what it does: continuing text plausibly.")

That is **hallucination**, and it's the same machinery that makes the model useful. There is no switch to flip. Big models hallucinate less and more convincingly, which is arguably worse - a fluent, confident, well-cited falsehood is harder to catch than obvious mush.

The rest of the list:

* **Bias.** Models learn from human text, so they learn human prejudice. A model trained on decades of hiring data learns who *used* to get hired. It's not neutral, it's an average of us, and averages of us have some ugly parts.
* **No idea what they don't know.** A model's confidence is not correlated with its correctness. It sounds precisely as certain when inventing a theorem as when reciting a real one.
* **Cost and energy.** Training a frontier model costs tens to hundreds of millions of pounds and consumes electricity at industrial scale. That concentrates the technology among a handful of organisations, which is a political fact as much as a technical one.
* **The data is running out.** These models are trained on a large fraction of the public internet. There is not a second internet. What happens next is an open question.
* **Stuck in the past.** A model knows the world as of its training data. It cannot know what happened yesterday unless you tell it or give it a search tool.

None of this means the technology doesn't work - you just watched it work. It means **"it sounds confident" is not evidence of anything**, and that the person checking the output is the important part of the system. Right now, that's you.

> **Going further:** [On the Dangers of Stochastic Parrots](https://dl.acm.org/doi/10.1145/3442188.3445922) is the most-cited critical paper in the field and worth reading even if you disagree with it. [Model cards](https://huggingface.co/docs/hub/model-cards) are where a model's own authors document its limitations, and reading one before you use a model is the professional habit almost nobody has.

# Where to Go Next

You now have the map. Every idea in this notebook has a notebook in this repo that goes deep on it:

* **Can't program yet, or want to be better at it?** `Learning_Python_In2Stem_Placement.ipynb` - the language everything above is written in.
* **Liked the digits, the confusion matrix, the overfitting curve?** `Intro_to_Data_Science_with_Python.ipynb` - NumPy, Pandas and Matplotlib properly, on real messy data.
* **Liked the tokenizer, attention, and text generation?** `Intro_to_NLP.ipynb` - text processing, sentiment analysis, and building chatbots on a real language model.
* **Liked "everything is numbers", the learned features, the neurons watching for strokes?** `Intro_to_computer_vision.ipynb` - images as data through to YOLO11 detection, segmentation, and generative image models.

The honest advice for what comes after: **build something that doesn't work, then find out why.** Everything in this notebook was written by people who did that. Reading about gradient descent is not the same as watching your own learning rate blow up, which is why we made you move the slider.

# Homework

Work through as many as you can. The stretch challenges are open-ended on purpose - there's no single right answer, and defending your reasoning matters more than the code.

**Warm-up, core computer science:**
1. **Secret messages**: use `text_to_bits` and `bits_to_text` to encode a message, then write a function that flips every bit (0 becomes 1). Decode it. Is the result still readable, and can you write the function that undoes it?
2. **Algorithm detective**: time Python's `sorted()` on lists of 1,000 / 10,000 / 100,000 / 1,000,000 random numbers. Plot the times. Is the shape closer to O(n), O(n log n), or O(n squared)? Does the measured shape match what the documentation claims?
3. **Your branch**: add the branch of computer science you'd most want to work in to the `BRANCHES` dictionary, properly filled in with a real job title and a real example. What convinced you it was that one?

**Core, machine learning:**
4. **A different dataset**: swap `load_digits` for `load_wine` or `load_breast_cancer` from `sklearn.datasets`. Train the same logistic regression, print the test accuracy, and draw the confusion matrix. Which classes does it confuse, and can you guess why from the feature names?
5. **Break the expert system**: take `triage_v2` and add rules until it has at least eight. Then find a case where two of *your own* rules contradict each other. How did you resolve it, and would the next person maintaining your code have resolved it the same way?
6. **A new logic gate**: train the from-scratch `Perceptron` on NOR (`[1, 0, 0, 0]`) and on "both inputs are the same" (`[1, 0, 0, 1]`). One will work and one will not. Predict which before running, then explain the result using the straight-line argument.
7. **Find the sweet spot**: in the overfitting section, change the number of data points from 18 to 200 and re-run the train-vs-test error plot. Where does the sweet spot move, and what does that tell you about why GPT-3 was trained on most of the internet?

**Stretch, go further:**
8. **Winter historian**: pick one of the two AI winters and research what actually happened - the Lighthill report for the first, the LISP machine collapse for the second. Write 200 words on whether you see the same pattern in AI today. Be specific: name the promise you think is being over-extrapolated.
9. **Hub explorer**: browse [huggingface.co/models](https://huggingface.co/models), find a `text-classification` model you haven't used, and run it through `pipeline()` on five sentences of your own. Read its model card first. Does it do what the card claims?
10. **The argument**: pick one limitation from the "What These Models Get Wrong" section and argue, in about 300 words, whether it's a temporary engineering problem or something fundamental to how these models work. Use evidence from this notebook. There is no right answer here and your reasoning is the entire point.

In [ ]:
# Warm-up Challenge 1: Secret messages
message = "your message here"  # TODO: your message

bits = text_to_bits(message)
print("Encoded:", bits[:60], "...")

# TODO: write flip_all_bits(bits) that turns every 0 into a 1 and every 1 into a 0.
# Careful: text_to_bits puts spaces between chunks. Don't flip the spaces.
# Then decode it. Then write the function that undoes it.

In [ ]:
# Warm-up Challenge 2: Algorithm detective
import random

sizes = [1_000, 10_000, 100_000, 1_000_000]
times = []

for n in sizes:
    data = [random.random() for _ in range(n)]
    start = time.perf_counter()
    sorted(data)
    times.append(time.perf_counter() - start)
    print(f"n = {n:9,}  ->  {times[-1] * 1000:8.2f} ms")

# TODO: plot sizes against times with matplotlib.
# TODO: overlay what pure O(n) and pure O(n log n) would look like, scaled to match
# your first measurement. Which one lies on top of your data?

In [ ]:
# Warm-up Challenge 3: Your branch
# TODO: fill this in honestly - a real job title, a real example that convinced you.
BRANCHES["Your Chosen Branch"] = {
    "studies": "",
    "question": "",
    "job": "",
    "example": "",
}

# TODO: re-run draw_field_map() and the dropdown to see yours on the map.

In [ ]:
# Core Challenge 4: A different dataset
from sklearn.datasets import load_wine

data = load_wine()   # TODO: try load_breast_cancer() too
print("Features:", data.feature_names)
print("Classes: ", data.target_names)
print("Shape:   ", data.data.shape)

# TODO: train_test_split, fit a LogisticRegression, print the test accuracy.
# TODO: draw the confusion matrix. Which classes get confused, and does that
# make sense given the feature names?

In [ ]:
# Core Challenge 5: Break the expert system
def triage_v3(symptoms):
    """Your expert system with at least eight rules. Find where it contradicts itself."""
    s = set(symptoms)

    # TODO: at least eight rules. Then find a case where two of yours both apply
    # and disagree. Write the case down as a comment and say how you resolved it.
    if "chest pain" in s and "shortness of breath" in s:
        return "EMERGENCY: call 999 immediately."
    return "No rule matched. Please see a GP."

print(triage_v3(["chest pain", "shortness of breath"]))

In [ ]:
# Core Challenge 6: A new logic gate
# TODO: predict FIRST. Sketch the four points. Can a straight line separate them?
y_nor = np.array([1, 0, 0, 0])          # NOR
y_same = np.array([1, 0, 0, 1])         # "both inputs are the same"

for name, targets in [("NOR", y_nor), ("SAME", y_same)]:
    neuron = Perceptron(n_inputs=2)
    neuron.train(X_and, targets, epochs=200)
    correct = sum(neuron.predict(xi) == t for xi, t in zip(X_and, targets))
    print(f"{name:5} -> got {correct}/4 correct")

# TODO: one of these fails. Explain why using the straight-line argument.

In [ ]:
# Core Challenge 7: Find the sweet spot
# TODO: regenerate the data with 200 points instead of 18, then re-run the
# train-vs-test error plot from the overfitting section.
n_points = 200

rng = np.random.RandomState(3)
X_big = np.sort(rng.uniform(0, 1, n_points))[:, None]
y_big = np.sin(2 * np.pi * X_big).ravel() + rng.normal(0, 0.25, n_points)

# TODO: loop over degrees 1-17, fit, record train and test error, and plot both.
# Where is the sweet spot now? Did the overfitting get better or worse with more data?

# Stretch Challenge 8 - write your answer here as markdown

In [ ]:
# Stretch Challenge 9: Hub explorer
from transformers import pipeline

# TODO: browse huggingface.co/models, filter by "Text Classification",
# read a model card, and paste the model id below.
classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english", device=-1)

sentences = [
    "This notebook was genuinely interesting.",
    # TODO: four more of your own. Try to find one that fools it.
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"{result['label']:8} {result['score']:.1%}  {sentence}")

# Stretch Challenge 10 - write your answer here as markdown

### References

- Turing, A. M. (1950). [Computing Machinery and Intelligence](https://academic.oup.com/mind/article/LIX/236/433/986238) - the paper that started it, and still readable in an afternoon.
- McCarthy, J. et al. (1955). [A Proposal for the Dartmouth Summer Research Project on Artificial Intelligence](http://www-formal.stanford.edu/jmc/history/dartmouth/dartmouth.html) - where the term was coined.
- Rosenblatt, F. (1958). [The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain](https://psycnet.apa.org/doi/10.1037/h0042519)
- Minsky, M. and Papert, S. (1969). *Perceptrons* - the book that caused the first AI winter.
- Krizhevsky, A., Sutskever, I. and Hinton, G. (2012). [ImageNet Classification with Deep Convolutional Neural Networks](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks) - AlexNet, the 2012 spike on our timeline.
- Vaswani, A. et al. (2017). [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - the Transformer.
- Bender, E. et al. (2021). [On the Dangers of Stochastic Parrots](https://dl.acm.org/doi/10.1145/3442188.3445922) - the best-known critical view of large language models.
- [scikit-learn documentation](https://scikit-learn.org/stable/) and its [user guide](https://scikit-learn.org/stable/user_guide.html)
- [Hugging Face Transformers documentation](https://huggingface.co/docs/transformers/index)
- [Hugging Face Model Hub](https://huggingface.co/models)
- [3Blue1Brown, Neural Networks](https://www.3blue1brown.com/topics/neural-networks) - the clearest explanation of backpropagation anywhere.
- [TensorFlow Playground](https://playground.tensorflow.org) - build and train a network in your browser.
- [Nand2Tetris](https://www.nand2tetris.org/) - build a whole computer from a single logic gate up.